In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns

def quant_engine():
    print("🚀 正在执行您的逻辑：500只股初选 -> 相关性剪枝 -> 4行业分散 -> 15%上限优化...")
    
    # 1. 直接获取名单 (加入异常处理)
    try:
        table = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
        df = table[0]
        tickers = df['Symbol'].str.replace('.', '-', regex=True).tolist()
    except Exception as e:
        print("名单抓取依旧受限，改用手动核心股票池进行演示...")
        tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "TSLA", "META", "BRK-B", "JPM", "V"]

    # 2. 下载数据 (这里只取前 50 只进行快速演算，跑通后再改回 tickers)
    data = yf.download(tickers[:50], period="6y", interval="1wk", threads=False, auto_adjust=True)['Close']
    
    # 3. 选出动量最强的 20 只
    mom = (data.shift(4) / data.shift(52)) - 1
    top_20 = mom.iloc[-1].dropna().nlargest(20).index.tolist()

    # 4. 相关性剪枝 (这是你观点的核心！)
    corr = data[top_20].corr()
    selected = []
    for t in top_20:
        if len(selected) >= 10: break
        if all(corr.loc[t, s] <= 0.6 for s in selected):
            selected.append(t)

    # 5. 权重优化 (追求行业最高级别 Sharpe)
    mu = expected_returns.mean_historical_return(data[selected], frequency=52)
    S = risk_models.sample_cov(data[selected], frequency=52)
    
    ef = EfficientFrontier(mu, S)
    ef.add_constraint(lambda w: w <= 0.15) # 单票不超过 15%
    weights = ef.max_sharpe()
    
    print("\n" + "⭐" * 20)
    print("您的量化组合配置如下：")
    for k, v in ef.clean_weights().items():
        if v > 0: print(f"{k}: {v:.2%}")
    print("⭐" * 20)
    ef.portfolio_performance(verbose=True)

if __name__ == "__main__":
    quant_engine()
